<table>
<tr>
<td><img src="https://www.institutobme.es/dam/layout/bme-logo.svg" width="150"></td>
<td><table><tr><td><h1>Carpetas y ficheros</h1></td></tr>
           <tr><td><h3>Rafael Caballero Roldán</h3></td></tr></table></td>

</tr>
</table>




### Índice
[Caminos](#Caminos)<br>
[Creación de carpetas](#Creación_de_carpetas)<br>
[Borrado](#Borrado)<br>
[Copia de ficheros](#Copia)<br>
[Ficheros .zip](#Zip)<br>

## Preparación

Ejecutar las dos siguientes celdas para asegurar que las bibliotecas que vamosa  usar están cargadas, y también para crear un par de ficheros de prueba

In [1]:
modules = ["pathlib", "shutil", "pandas", "numpy","pyarrow"]

import sys
import os.path
from subprocess import check_call
import importlib
import os

def instala(modules):
  print("Instalando módulos")
  for m in modules:
      # para el import quitamos [...] y ==...
      p = m.find("[")
      mi = m if p==-1 else m[:p]
      p = mi.find("==")
      mi = mi if p==-1 else mi[:p]

      torch_loader = importlib.util.find_spec(mi)
      if torch_loader is not None:
          print(m," encontrado")
      else:
          print(m," No encontrado, instalando...",end="")  
          try:        
            r = check_call([sys.executable, "-m", "pip", "install", "--user", m])
            print("¡hecho!")
          except:
            print("¡Problema al instalar ",m,"! ¿seguro que el módulo existe?",sep="")
              
  print("¡Terminado!")

instala(modules) 

Instalando módulos
pathlib  encontrado
shutil  encontrado
pandas  encontrado
numpy  encontrado
pyarrow  encontrado
¡Terminado!


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
path = Path.cwd() 

# datos
numcolumnas = 100
filas = []
numfilas = 10000
for i in range(numfilas):
    filas.append(np.random.randint(1,5000,numcolumnas))

df = pd.DataFrame(filas, columns = ["c"+str(i) for i in range(numcolumnas)])
fichero_csv = path / "prueba.csv"
fichero_parquet = path / "prueba.parquet"
df.to_csv(fichero_csv,index=False)
df.to_parquet(fichero_parquet, index=False)

<a name="Caminos"></a>
#### Caminos

En cualquier sistema operativo los sistemas de almacenamiento permanentes (discusos, memorias flash, etc) van a tener un sistema de carpetas y ficheros:
    
    * Las carpetas pueden contener ficheros y otras carpetas
    * Los ficheros contienen datos


El camino a un fichero se puede dar de dos formas:

#### Camino Absoluto

Ejemplo  **d:\docencia\2526\apd\codigo\path.ipynb**

Estos caminos comienzan generalmente por el nombre de unidad, luego la secuencia de carpetas y finalmente el nombre del fichero con su extensión. 

En el caso de Windows el separador de elementos es \ y en el caso de linux es /. Como la \ es un símbolo especial suele dar problemas, por eso Python incluye el prefijo r para poder usarlo:

        camino = r"c:\bertoldo\ficheros\datos.csv"

#### Camino Relativo

Se mueve desde el lugar en el que estamos (en el caso de un notebook sería desde el lugar donde está el notebook). Se utilia
- . Para indicar la carpeta actual como  **.\img\log.png**: que significa: busca dentro de la carpeta actual (.) una carpeta img y dentro de img el fichero log.png
- .. Para indicar el antecesor de la carpetaa anterior. Por ejemplo **..\datos\trafico.csv**, que indica que la carpeta datos es "hermana" de la actual

    
Debemos ser capaces de manejar estas estructuras en Python para automatizar nuestro proceso de adquisición de datos.

Primero veamos en qué carpeta nos encontramos ahora mismo (la del notebook de Python)

In [3]:
from pathlib import Path
path = Path.cwd()  
print(path)

c:\rafa\docencia\2526\apd\codigo


Añadimos un fichero al final del camino

In [4]:
import pandas as pd
df = pd.DataFrame({"nombre":["Bertoldo", "Herminia","Calixta","Aniceto"],"edad":[18,19,24,30]})
p =  path / "datos.csv"             # Alternativa: Path(path,"datos.csv")
df.to_csv(p,index=False)
print(p)

c:\rafa\docencia\2526\apd\codigo\datos.csv


¿El path existe? ¿es una carpeta o un fichero?

In [5]:
if p.exists():
    print(p,"existe",end=" ")
    if p.is_dir():
        print("y es una carpeta")
    else:
        if p.is_file():
            print("y es un fichero")
else:
    print(p,"no existe")

c:\rafa\docencia\2526\apd\codigo\datos.csv existe y es un fichero


Ya sabemos que p es un camino a un fichero, ahora vamos a ver algunas de sus características

In [6]:
p.absolute

<bound method Path.absolute of WindowsPath('c:/rafa/docencia/2526/apd/codigo/datos.csv')>

In [7]:
p.drive

'c:'

In [8]:
p.parent

WindowsPath('c:/rafa/docencia/2526/apd/codigo')

In [9]:
p.name

'datos.csv'

In [10]:
p.stem

'datos'

In [11]:
p.suffix

'.csv'

Además podemos obtener [algunos datos](https://docs.python.org/3/library/os.html#os.stat_result) del fichero, por ejemplo su tamaño en bytes

In [12]:
estad = p.stat() # estadísticas varias
estad

os.stat_result(st_mode=33206, st_ino=72902018968060445, st_dev=4655120325201049254, st_nlink=1, st_uid=0, st_gid=0, st_size=63, st_atime=1771924148, st_mtime=1771924148, st_ctime=1771841335)

In [13]:
estad.st_size 

63

¿Y si no sé cómo se llama el fichero?

Todas las carpetas y ficheros del path actual (que debe ser una carpeta)

In [14]:
def lista(path):
    print(path)
    espacio = " "*5
    for p in path.iterdir():
        if p.is_dir():
            print(espacio,p.name,": carpeta",sep="")
        else:
            if p.is_file():
                print(espacio,p.name,": fichero",sep="")
    print("="*40)
                
lista(path)                

c:\rafa\docencia\2526\apd\codigo
     accidentes.xlsx: fichero
     BeautifulSoup.ipynb: fichero
     carga.ipynb: fichero
     control.ipynb: fichero
     dataframes.ipynb: fichero
     dataframesCompletado.ipynb: fichero
     datos.csv: fichero
     estructuras.ipynb: fichero
     examenes1D.pdf: fichero
     examenes26.pdf: fichero
     fechas.ipynb: fichero
     informe24.pdf: fichero
     intro.ipynb: fichero
     numpy.ipynb: fichero
     pagina.html: fichero
     paises.csv: fichero
     parocomunidades.csv: fichero
     paths.ipynb: fichero
     prueba.csv: fichero
     prueba.parquet: fichero
     raw: carpeta
     resumen.pdf: fichero
     selenium.ipynb: fichero
     series.ipynb: fichero
     web.ipynb: fichero


También se puede usar `glob` que tiene alguna ventaja:

- Permite obtener solo los ficheros/carpetas que cumplan un patrón
- Tiene una versión recursiva que busca también en los hijos (ojo, esto puede hacer que sea lento si se hace en un sistema de ficheros grande)

In [15]:
from pathlib import Path

root = Path.cwd()

# todos los ficheros de la carpeta actual
files = [p for p in root.glob("*") if p.is_file()]

# todos los ficheros de la carpeta actual y descendientes (recursivo)
rfiles = [p for p in root.rglob("*") if p.is_file()]

# solo CSV 
csvs = list(root.glob("*.csv"))

print(f"Esta carpeta tiene ficheros{len(files)} ficheros ({len(rfiles)} si contamos descendientes), de los cuales {len(csvs)} son csvs")

Esta carpeta tiene ficheros24 ficheros (39 si contamos descendientes), de los cuales 4 son csvs


In [30]:
csvs = list(root.glob("./raw/*.csv"))
csvs

[WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/abr_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/ago_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/dic_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/ene_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/feb_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/jul_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/jun_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/mar_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/may_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/nov_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/oct_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/sep_meteo22.csv')]

In [35]:
dfs =  [  pd.read_csv(f,sep=";") for f in csvs]
print(len(dfs))

12


In [34]:
dfs[0]

,PROVINCIA,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,DIA,H01,V01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
0,28,79,102,81,28079102_81_98,2022,4,1,2.17,V,...,1.72,V,2.83,V,2.12,V,0.00,N,0.00,N
1,28,79,102,81,28079102_81_98,2022,4,2,2.70,V,...,3.42,V,2.82,V,2.78,V,3.30,V,0.00,N
2,28,79,102,81,28079102_81_98,2022,4,3,2.22,V,...,4.75,V,0.00,N,0.00,N,0.00,N,0.00,N
3,28,79,102,81,28079102_81_98,2022,4,4,5.72,V,...,1.88,V,1.57,V,0.48,V,1.28,V,3.15,V
4,28,79,102,81,28079102_81_98,2022,4,5,2.60,V,...,2.93,V,2.63,V,2.80,V,2.68,V,2.05,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2714,28,79,59,89,28079059_89_98,2022,4,26,0.00,V,...,0.30,V,0.10,V,0.10,V,0.30,V,0.20,V
2715,28,79,59,89,28079059_89_98,2022,4,27,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V
2716,28,79,59,89,28079059_89_98,2022,4,28,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V
2717,28,79,59,89,28079059_89_98,2022,4,29,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V


In [36]:
df_grande = pd.concat(dfs)
df_grande

,PROVINCIA,MUNICIPIO,ESTACION,MAGNITUD,PUNTO_MUESTREO,ANO,MES,DIA,H01,V01,...,H20,V20,H21,V21,H22,V22,H23,V23,H24,V24
0,28,79,102,81,28079102_81_98,2022,4,1,2.17,V,...,1.72,V,2.83,V,2.12,V,0.00,N,0.00,N
1,28,79,102,81,28079102_81_98,2022,4,2,2.70,V,...,3.42,V,2.82,V,2.78,V,3.30,V,0.00,N
2,28,79,102,81,28079102_81_98,2022,4,3,2.22,V,...,4.75,V,0.00,N,0.00,N,0.00,N,0.00,N
3,28,79,102,81,28079102_81_98,2022,4,4,5.72,V,...,1.88,V,1.57,V,0.48,V,1.28,V,3.15,V
4,28,79,102,81,28079102_81_98,2022,4,5,2.60,V,...,2.93,V,2.63,V,2.80,V,2.68,V,2.05,V
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1255,28,79,59,89,28079059_89_98,2022,9,26,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V
1256,28,79,59,89,28079059_89_98,2022,9,27,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V
1257,28,79,59,89,28079059_89_98,2022,9,28,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V
1258,28,79,59,89,28079059_89_98,2022,9,29,0.00,V,...,0.00,V,0.00,V,0.00,V,0.00,V,0.00,V


<a name="Creación_de_carpetas"></a>
#### Creación de carpetas

Es un buen hábito tener los datos iniciarles en una carpeta `raw` y dejar los resultados en una carpeta `preprocess`. Para ello debemos ser capaces de crear carpetas si no existen

In [38]:
from pathlib import Path
path = Path.cwd()  
pathraw = Path(path,"../misdatos")
if pathraw.exists():
    print("Ya existe")
else:
    print("No existe, lo creamos")
    pathraw.mkdir()
    if pathraw.exists():
        print("Creado")
    else:
        print("No se puede crear!!")

No existe, lo creamos
Creado


In [40]:
from pathlib import Path
path = Path.cwd()  
for i in range(500):
    pathraw = Path(path,"../misdatos/datos"+str(i))
    if not pathraw.exists():
        pathraw.mkdir()
        

Si ya existe y no lo comprobamos  da error

In [17]:
#pathraw.mkdir() Si quitamos el comentario

Pero se puede hacer mejor con el atributo exists_ok, que indica que si ya existe no haga nada (y si no lo crea)

In [18]:
pathraw.mkdir(exist_ok=True)

Otro parámetro util para `mkdir` es parents=True, que creará además todos las carpetas intermedias. Esto es muy útil para crear una estructura compleja de ficheros

In [ ]:
folders = ["tururu/download/feb","misdatos/download/mar","misdatos/download/may"] 
for f in folders:
    newFolder = Path(path,f)
    newFolder.mkdir(exist_ok=True,parents=True)

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'c:\\rafa\\docencia\\2526\\apd\\codigo\\tururu\\download\\feb'

In [45]:
carpeta_inicial = Path.cwd()
carpeta = carpeta_inicial
for i in range(500):
    carpeta = carpeta / ("raw"+str(i))


carpeta

WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw0/raw1/raw2/raw3/raw4/raw5/raw6/raw7/raw8/raw9/raw10/raw11/raw12/raw13/raw14/raw15/raw16/raw17/raw18/raw19/raw20/raw21/raw22/raw23/raw24/raw25/raw26/raw27/raw28/raw29/raw30/raw31/raw32/raw33/raw34/raw35/raw36/raw37/raw38/raw39/raw40/raw41/raw42/raw43/raw44/raw45/raw46/raw47/raw48/raw49/raw50/raw51/raw52/raw53/raw54/raw55/raw56/raw57/raw58/raw59/raw60/raw61/raw62/raw63/raw64/raw65/raw66/raw67/raw68/raw69/raw70/raw71/raw72/raw73/raw74/raw75/raw76/raw77/raw78/raw79/raw80/raw81/raw82/raw83/raw84/raw85/raw86/raw87/raw88/raw89/raw90/raw91/raw92/raw93/raw94/raw95/raw96/raw97/raw98/raw99/raw100/raw101/raw102/raw103/raw104/raw105/raw106/raw107/raw108/raw109/raw110/raw111/raw112/raw113/raw114/raw115/raw116/raw117/raw118/raw119/raw120/raw121/raw122/raw123/raw124/raw125/raw126/raw127/raw128/raw129/raw130/raw131/raw132/raw133/raw134/raw135/raw136/raw137/raw138/raw139/raw140/raw141/raw142/raw143/raw144/raw145/raw146/raw147/raw148/raw149/raw150/raw151/

In [46]:
carpeta.mkdir(parents=True)

FileNotFoundError: [WinError 206] El nombre del archivo o la extensión es demasiado largo: 'c:\\rafa\\docencia\\2526\\apd\\codigo\\raw0\\raw1\\raw2\\raw3\\raw4\\raw5\\raw6\\raw7\\raw8\\raw9\\raw10\\raw11\\raw12\\raw13\\raw14\\raw15\\raw16\\raw17\\raw18\\raw19\\raw20\\raw21\\raw22\\raw23\\raw24\\raw25\\raw26\\raw27\\raw28\\raw29\\raw30\\raw31\\raw32\\raw33\\raw34\\raw35\\raw36\\raw37'

<a name="Borrado"></a>
#### Borrado de carpetas y ficheros

En principio disponemos de dos métodos interesantes   
    
    * path.unlink() borra un fichero
    * path.rmdir() borra una carpeta

In [47]:
from pathlib import Path
path = Path.cwd() 
print("Antes")
lista(path)
fichero_csv = Path(path,"prueba.csv")
fichero_csv.unlink(missing_ok=True) # no da error aunque no exista
print("Después")
lista(path)

Antes
c:\rafa\docencia\2526\apd\codigo
     accidentes.xlsx: fichero
     BeautifulSoup.ipynb: fichero
     carga.ipynb: fichero
     control.ipynb: fichero
     dataframes.ipynb: fichero
     dataframesCompletado.ipynb: fichero
     datos.csv: fichero
     estructuras.ipynb: fichero
     examenes1D.pdf: fichero
     examenes26.pdf: fichero
     fechas.ipynb: fichero
     informe24.pdf: fichero
     intro.ipynb: fichero
     misdatos: carpeta
     numpy.ipynb: fichero
     pagina.html: fichero
     paises.csv: fichero
     parocomunidades.csv: fichero
     paths.ipynb: fichero
     prueba.parquet: fichero
     raw: carpeta
     resumen.pdf: fichero
     selenium.ipynb: fichero
     series.ipynb: fichero
     web.ipynb: fichero
Después
c:\rafa\docencia\2526\apd\codigo
     accidentes.xlsx: fichero
     BeautifulSoup.ipynb: fichero
     carga.ipynb: fichero
     control.ipynb: fichero
     dataframes.ipynb: fichero
     dataframesCompletado.ipynb: fichero
     datos.csv: fichero
     est

En el caso de borrar una carpeta debe estar vacía:

In [49]:
from pathlib import Path
path = Path.cwd() 
raw_dir = path / "raw"
raw_dir.rmdir() #si quitamos el comentario dará error

OSError: [WinError 145] El directorio no está vacío: 'c:\\rafa\\docencia\\2526\\apd\\codigo\\raw'

In [50]:
fichs = list(root.glob("./raw/*"))
fichs


[WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/abr_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/ago_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/dic_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/download'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/ene_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/feb_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/Interpretación_datos_meteorologicos.pdf'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/jul_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/jun_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/mar_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/may_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/nov_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/oct_meteo22.csv'),
 WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw/readme.txt'),
 Windows

Si no queremos borrar elemento a elemento y queremos borrar todo un subárbol podemos usar la librería `shutil`

OJO: esto puede borrar el disco duro entero sin preguntar, usar con precaución

In [51]:
raw_dir

WindowsPath('c:/rafa/docencia/2526/apd/codigo/raw')

In [52]:
from shutil import rmtree
rmtree(raw_dir,ignore_errors=True) # para que no dé error si no existe

<a name="Copia"></a>
#### Copia de ficheros
Esta biblioteca también nos permite copiar ficheros de un destino a otro

Primero creamos algunas carpetas y un fichero de prueba 

In [53]:
folders = ["raw/download/feb","raw/download/mar","raw/download/may"] 
for f in folders:
    newFolder = path / f
    newFolder.mkdir(exist_ok=True,parents=True)
    
import pandas as pd
df = pd.DataFrame({"nombre":["Bertoldo", "Herminia","Calixta","Aniceto"],"edad":[18,19,24,30]})
p = Path(folders[0]) / "datos.csv"
df.to_csv(p,index=False)
lista(Path(folders[0]))  

raw\download\feb
     datos.csv: fichero


In [55]:
import pathlib
import shutil

print("Antes")
lista(Path(folders[1]))
origen=p
for i in range(500):
    destino= Path(folders[1]) / ("datoscopia"+str(i)+".csv")
    print(origen,destino)
    shutil.copy(origen, destino)
print("después")
lista(Path(folders[1]))


Antes
raw\download\mar
     datoscopia.csv: fichero
raw\download\feb\datos.csv raw\download\mar\datoscopia0.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia1.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia2.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia3.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia4.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia5.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia6.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia7.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia8.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia9.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia10.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia11.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia12.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia13.csv
raw\download\feb\datos.csv raw\download\mar\datoscopia14.csv
raw\download\feb\datos.csv raw\download\mar

In [56]:
from shutil import rmtree
rmtree(raw_dir,ignore_errors=True) 

<a name="Zip"></a>
#### Ficheros .zip


Primero creamos algunas carpetas y un fichero de prueba 

In [58]:
from pathlib import Path
path = Path.cwd() 
download_dir = Path(path,"raw/download")
download_dir.mkdir(exist_ok=True,parents=True)

fichero = "meteo22.zip"
path_fich = Path(download_dir,fichero)

In [59]:
import requests
url = "https://github.com/RafaelCaballero/tdm/blob/master/datos/madrid/meteo22.zip?raw=true"
r = requests.get(url)
if r.status_code==200:
    with open(path_fich, 'wb') as f:
        f.write(r.content) # ahora lo grabamos localmente
    print("Grabado")
else:
    print("Error ",r.status_code)
    
lista(download_dir)    


Grabado
c:\rafa\docencia\2526\apd\codigo\raw\download
     meteo22.zip: fichero


Utilizamos `unpack_archive` que detecta el tipo de compresión y descomprime automáticamente. El primer parámetro es el fichero y el segundo la carpeta donde debe guardarse

In [60]:
import shutil
raw_dir = Path(path,"raw")
shutil.unpack_archive(path_fich, raw_dir)
lista(raw_dir)

c:\rafa\docencia\2526\apd\codigo\raw
     abr_meteo22.csv: fichero
     ago_meteo22.csv: fichero
     dic_meteo22.csv: fichero
     download: carpeta
     ene_meteo22.csv: fichero
     feb_meteo22.csv: fichero
     Interpretación_datos_meteorologicos.pdf: fichero
     jul_meteo22.csv: fichero
     jun_meteo22.csv: fichero
     mar_meteo22.csv: fichero
     may_meteo22.csv: fichero
     nov_meteo22.csv: fichero
     oct_meteo22.csv: fichero
     readme.txt: fichero
     sep_meteo22.csv: fichero


Para acabar un video corto con algunas ideas sobre nombres y organización de ficheros [https://www.youtube.com/watch?v=YslfY4W-NAg](https://www.youtube.com/watch?v=YslfY4W-NAg)